# Project Setup and DVC Workflow

Run this notebook from the repository root. It:

1. Writes `params.yaml`, `dvc.yaml`, `requirements.txt`, `.gitignore` and `README.md`
2. Initialises DVC (`dvc init`) and adds a local remote
3. Runs the full pipeline with `dvc repro`
4. Shows `dvc status`, `dvc dag` and `dvc metrics show`
5. Uploads data and models with `dvc push`

Install the requirements first: `pip install -r requirements.txt` (or `pip install pandas numpy scikit-learn scipy matplotlib seaborn pyyaml joblib jupyter nbconvert ipykernel dvc`).

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

env = os.environ.copy()
bin_dirs = [str(Path(sys.executable).parent), str(Path(sys.executable).parent / 'Scripts')]
env['PATH'] = os.pathsep.join(bin_dirs + [env.get('PATH', '')])


def run(*args):
    print('$', ' '.join(args))
    result = subprocess.run(list(args), capture_output=True, text=True, env=env)
    print(result.stdout)
    if result.stderr.strip():
        print(result.stderr)
    return result.returncode


print('Working directory:', Path.cwd())

## 1. Parameters: `params.yaml`

Every tunable value of the pipeline. DVC tracks these values, so editing one makes `dvc repro` rerun only the stages that use it.

In [ ]:
PARAMS_YAML = """\
base:
  random_state: 42
  target: is_canceled

data:
  raw: hotel_bookings.csv
  processed_dir: data/processed

prepare:
  test_size: 0.2
  iqr_multiplier: 1.5
  clip_columns:
    - adr
    - Stay_Duration
  drop_columns:
    - reservation_status
    - reservation_status_date
    - name
    - email
    - phone-number
    - credit_card

train:
  n_estimators: 100
  max_depth: 20
  min_samples_split: 5
  min_samples_leaf: 2
  class_weight: null

evaluate:
  threshold: 0.5

drift:
  reference_years: [2015, 2016]
  current_years: [2017]
  exclude_columns:
    - arrival_date_year
    - arrival_date_month
    - arrival_date_week_number
    - arrival_date_day_of_month
  psi_bins: 10
  psi_threshold: 0.2
  drifted_share_threshold: 0.3
  max_f1_drop: 0.05
"""

Path("params.yaml").write_text(PARAMS_YAML, encoding="utf-8")
print("Wrote params.yaml")

## 2. Pipeline definition: `dvc.yaml`

Each stage lists the command DVC runs (execute a notebook), its dependencies (`deps`), the parameters it reads (`params`) and the files it produces (`outs`, `metrics`, `plots`).

In [ ]:
DVC_YAML = """\
stages:
  prepare:
    cmd: jupyter nbconvert --to notebook --execute prepare.ipynb --output-dir executed
    deps:
      - prepare.ipynb
      - hotel_bookings.csv
    params:
      - base
      - data
      - prepare
    outs:
      - data/processed/train.csv
      - data/processed/test.csv
      - models/preprocessor.pkl

  train:
    cmd: jupyter nbconvert --to notebook --execute train.ipynb --output-dir executed
    deps:
      - train.ipynb
      - data/processed/train.csv
    params:
      - base
      - data
      - train
    outs:
      - models/model.pkl

  evaluate:
    cmd: jupyter nbconvert --to notebook --execute evaluate.ipynb --output-dir executed
    deps:
      - evaluate.ipynb
      - models/model.pkl
      - data/processed/test.csv
    params:
      - base
      - data
      - evaluate
    metrics:
      - metrics/metrics.json:
          cache: false
    plots:
      - plots/roc_curve.csv:
          cache: false
          x: fpr
          y: tpr
      - plots/confusion_matrix.png:
          cache: false
      - plots/feature_importance.png:
          cache: false

  drift:
    cmd: jupyter nbconvert --to notebook --execute drift_detection.ipynb --output-dir executed
    deps:
      - drift_detection.ipynb
      - data/processed/train.csv
      - data/processed/test.csv
    params:
      - base
      - data
      - train
      - drift
    outs:
      - metrics/drift_features.csv:
          cache: false
    metrics:
      - metrics/drift_report.json:
          cache: false
    plots:
      - plots/drift_psi.png:
          cache: false
"""

Path("dvc.yaml").write_text(DVC_YAML, encoding="utf-8")
print("Wrote dvc.yaml")

## 3. `requirements.txt` and `.gitignore`

In [ ]:
REQUIREMENTS = """\
pandas
numpy
scikit-learn
scipy
matplotlib
seaborn
pyyaml
joblib
jupyter
nbconvert
ipykernel
dvc
"""

Path("requirements.txt").write_text(REQUIREMENTS, encoding="utf-8")
print("Wrote requirements.txt")

GITIGNORE = """\
executed/
dvc_plots/
.ipynb_checkpoints/
__pycache__/
.venv/
venv/
"""

Path(".gitignore").write_text(GITIGNORE, encoding="utf-8")
print("Wrote .gitignore")

## 4. `README.md`

In [ ]:
README = """\
MLSD Project: Hotel Booking Cancellation Prediction
===================================================

DS-4491 Machine Learning Systems Design project. A reproducible machine learning pipeline, managed with **DVC**,
that predicts whether a hotel booking will be cancelled. Every pipeline stage is a Jupyter notebook that DVC
executes, so the whole experiment can be rebuilt with a single `dvc repro`.

Problem and Dataset
-------------------

**Problem:** hotels lose revenue when bookings are cancelled late. If a hotel can predict which bookings are
likely to be cancelled, it can overbook safely, ask for deposits or contact risky guests in advance.
This is a **binary classification** task: predict `is_canceled` (1 = cancelled, 0 = not cancelled).

**Dataset:** `hotel_bookings.csv`, the Hotel Booking Demand dataset (Antonio, Almeida and Nunes, 2019).

| Property | Value |
|----------|-------|
| Rows (raw) | 119,390 bookings |
| Columns | 32 |
| Hotels | City Hotel and Resort Hotel (Portugal) |
| Period | Arrivals from July 2015 to August 2017 |
| Target | `is_canceled` |
| Rows after cleaning | 87,229 |
| Cancellation rate after cleaning | 27.5% |

Preprocessing (`prepare` stage):

- Fill missing `children`, `country`, `agent`, `company`
- Remove duplicate bookings
- Drop personal data columns (`name`, `email`, `phone-number`, `credit_card`) if present
- Drop `reservation_status` and `reservation_status_date`, because they are recorded **after** the outcome and would leak the target
- Feature engineering: `Total_Guests`, `Stay_Duration`, month name to month number
- Remove invalid rows (negative price or zero guests)
- Stratified 80/20 train/test split
- IQR clipping of `adr` and `Stay_Duration` (bounds learned on the training set only)
- Ordinal encoding of categorical columns (fitted on the training set only)

ML Model
--------

**Random Forest Classifier** (scikit-learn). It handles mixed numeric/categorical features without scaling,
captures non-linear effects such as lead time and special requests, trains in a few seconds and gives feature
importances that are easy to explain. All hyperparameters live in `params.yaml`:

| Parameter | Value |
|-----------|-------|
| `n_estimators` | 100 |
| `max_depth` | 20 |
| `min_samples_split` | 5 |
| `min_samples_leaf` | 2 |
| `random_state` | 42 |

`hotel_booking.ipynb` contains the full exploratory analysis (EDA, feature importance, model comparison
between Decision Tree, Logistic Regression, Random Forest and Gradient Boosting, PCA and business applications)
that led to this choice.

Project Structure
-----------------

```
MLSD_Project/
├── hotel_bookings.csv          raw dataset
├── hotel_booking.ipynb         exploratory analysis and model comparison
├── setup_project.ipynb         generates the config files and runs the DVC workflow
├── prepare.ipynb               stage 1: cleaning, feature engineering, split, encoding
├── train.ipynb                 stage 2: Random Forest training
├── evaluate.ipynb              stage 3: metrics and plots on the test set
├── drift_detection.ipynb       stage 4 (bonus): data and concept drift detection
├── params.yaml                 all parameters
├── dvc.yaml                    pipeline definition
├── dvc.lock                    exact hashes of every dependency and output (created by dvc repro)
├── requirements.txt
├── .dvc/                       DVC configuration (created by dvc init)
├── data/processed/             train.csv, test.csv            (DVC output)
├── models/                     preprocessor.pkl, model.pkl    (DVC output)
├── metrics/                    metrics.json, drift_report.json, drift_features.csv
└── plots/                      roc_curve.csv, confusion_matrix.png, feature_importance.png, drift_psi.png
```

`data/processed/` and `models/` are **not** stored in Git. DVC caches them and `dvc push` uploads them to the
DVC remote. Git only stores the small `dvc.lock` file that records their hashes.

DVC Pipeline
------------

```
hotel_bookings.csv → prepare → train → evaluate → metrics / plots
                          └──────→ drift → drift report
```

```mermaid
flowchart LR
    A[hotel_bookings.csv] --> B[prepare]
    B -->|train.csv| C[train]
    B -->|test.csv| D[evaluate]
    C -->|model.pkl| D
    D --> E[metrics.json + plots]
    B -->|train.csv + test.csv| F[drift]
    F --> G[drift_report.json]
```

| Stage | Command (runs the notebook) | Dependencies | Parameters | Outputs |
|-------|-----------------------------|--------------|------------|---------|
| `prepare` | `jupyter nbconvert --execute prepare.ipynb` | `prepare.ipynb`, `hotel_bookings.csv` | `base`, `data`, `prepare` | `data/processed/train.csv`, `data/processed/test.csv`, `models/preprocessor.pkl` |
| `train` | `jupyter nbconvert --execute train.ipynb` | `train.ipynb`, `train.csv` | `base`, `data`, `train` | `models/model.pkl` |
| `evaluate` | `jupyter nbconvert --execute evaluate.ipynb` | `evaluate.ipynb`, `model.pkl`, `test.csv` | `base`, `data`, `evaluate` | `metrics/metrics.json`, `plots/*` |
| `drift` | `jupyter nbconvert --execute drift_detection.ipynb` | `drift_detection.ipynb`, `train.csv`, `test.csv` | `base`, `data`, `train`, `drift` | `metrics/drift_report.json`, `metrics/drift_features.csv`, `plots/drift_psi.png` |

Output of `dvc dag`:

```
                  +---------+              
                  | prepare |              
                **+---------+**            
             ***       *       ****        
          ***          *           ***     
        **             *              ***  
+-------+         +-------+              **
| drift |         | train |             ** 
+-------+         +-------+            *   
                           *         **    
                            **     **      
                              *   *        
                          +----------+     
                          | evaluate |     
                          +----------+
```

How to Run
----------

**1. Clone and install**

```bash
git clone https://github.com/Oney-999/MLSD_Project.git
cd MLSD_Project
python -m venv venv
source venv/bin/activate            (Windows: venv/Scripts/activate)
pip install -r requirements.txt
```

**2a. Run everything from the notebook**

Open `setup_project.ipynb` and run all cells. It writes `params.yaml`, `dvc.yaml`, `requirements.txt`,
`.gitignore` and this README, initialises DVC, adds a local remote, runs `dvc repro`, shows `dvc status`,
`dvc dag` and `dvc metrics show`, and finally runs `dvc push`.

**2b. Or run the DVC commands yourself**

```bash
dvc init                                   only once per repository
dvc remote add -d storage ../dvc_storage   local folder used as DVC remote
dvc repro                                  run the full pipeline
dvc status                                 check whether anything changed
dvc dag                                    show the pipeline graph
dvc metrics show                           show metrics
dvc plots show                             build dvc_plots/index.html
dvc push                                   upload data and model to the remote
```

**3. Commit the pipeline to Git**

```bash
git add dvc.yaml dvc.lock params.yaml .dvc .gitignore metrics plots README.md requirements.txt
git commit -m "Run DVC pipeline"
git push
```

**4. Reproduce on another machine**

```bash
git clone https://github.com/Oney-999/MLSD_Project.git
cd MLSD_Project
pip install -r requirements.txt
dvc pull          download the exact data and model from the remote
dvc repro         nothing reruns if everything matches dvc.lock
```

**Try a change**

- Edit `train.max_depth` in `params.yaml`, then `dvc status` shows `train` as changed and `dvc repro` reruns only `train`, `evaluate` and `drift` (`prepare` is skipped).
- Compare with the previous run using `dvc metrics diff` and `dvc params diff`.
- Change `hotel_bookings.csv` and every stage reruns, because the file hash in `dvc.lock` no longer matches.

**Tracking the dataset with `dvc add`**

The raw CSV is stored in Git in this repository so the course repo is self-contained. To track it with DVC instead:

```bash
git rm --cached hotel_bookings.csv
dvc add hotel_bookings.csv                 creates hotel_bookings.csv.dvc
git add hotel_bookings.csv.dvc .gitignore
git commit -m "Track dataset with DVC"
dvc push
```

Results
-------

Random Forest on the held-out test set (17446 bookings, threshold 0.5):

| Metric | Score |
|--------|-------|
| Accuracy | 0.8491 |
| Precision | 0.7798 |
| Recall | 0.6297 |
| F1 | 0.6968 |
| ROC AUC | 0.9145 |

The most important features are `lead_time`, `country`, `total_of_special_requests`, `agent` and
`market_segment` (see `plots/feature_importance.png`).

![Confusion matrix](plots/confusion_matrix.png)
![Feature importance](plots/feature_importance.png)

**Drift detection (bonus)**

Reference window: arrivals in 2015–2016 (55597 bookings). Current window: arrivals in 2017
(31632 bookings).

| Check | Result |
|-------|--------|
| Data drift (PSI > 0.2) | 0 of 27 features, max PSI 0.1403 |
| Target drift (cancellation rate) | 0.2501 → 0.3195 |
| Concept drift (F1 of a 2015–2016 model) | 0.7106 on reference → 0.5552 on 2017 (drop 0.1554) |
| Retrain recommended | True |

Individual feature distributions stay fairly stable, but a model trained only on older bookings loses a lot of F1
on 2017 bookings and the cancellation rate rises. The relationship between features and cancellations has changed
(concept drift), so the response is to retrain on recent data: update the dataset and run `dvc repro`.

![Drift PSI](plots/drift_psi.png)

DVC Command Reference
---------------------

| Command | What it does |
|---------|--------------|
| `dvc init` | Creates the `.dvc/` folder and turns the Git repo into a DVC project |
| `dvc add <file>` | Starts tracking a data file: moves it to the DVC cache, creates `<file>.dvc` and adds the file to `.gitignore` |
| `dvc repro` | Runs the stages in `dvc.yaml` whose dependencies, parameters or outputs changed, then updates `dvc.lock` |
| `dvc status` | Compares the workspace with `dvc.lock` and lists stages that are out of date |
| `dvc dag` | Prints the stage graph defined in `dvc.yaml` |
| `dvc push` | Uploads cached data and models to the DVC remote |
| `dvc pull` | Downloads the data and models recorded in `dvc.lock` / `.dvc` files from the remote |
| `dvc metrics show` / `diff` | Shows or compares metrics files |
| `dvc params diff` | Shows parameter changes compared with the last commit |

**How DVC knows which stage to rerun:** `dvc.lock` stores an MD5 hash for every dependency, parameter value and
output. `dvc repro` recomputes the hashes and only reruns a stage (and the stages after it) when something differs.

**Git vs DVC:** Git versions code and small text files (notebooks, `dvc.yaml`, `params.yaml`, `dvc.lock`).
DVC versions large data and model files by storing only their hashes in Git and the files themselves in a cache or remote.
"""

Path("README.md").write_text(README, encoding="utf-8")
print("Wrote README.md")

## 5. `dvc init`

Creates the `.dvc/` folder. Only needed once; skipped if DVC is already initialised.

In [ ]:
if not Path('.git').exists():
    run('git', 'init')
if not Path('.dvc').exists():
    run('dvc', 'init')
else:
    print('DVC already initialised')

## 6. DVC remote

A local folder next to the repository acts as the DVC remote storage for `dvc push` / `dvc pull`. It could be replaced with Google Drive, S3 or any other supported remote.

In [ ]:
remote_dir = Path('..') / 'dvc_storage'
remote_dir.mkdir(exist_ok=True)
run('dvc', 'remote', 'add', '-d', '-f', 'storage', str(remote_dir))
run('dvc', 'remote', 'list')

## 7. `dvc repro`

Runs every stage whose dependencies or parameters changed since the last run.

In [ ]:
run('dvc', 'repro')

## 8. `dvc status`

After a successful `dvc repro` this reports that the pipeline is up to date.

In [ ]:
run('dvc', 'status')

## 9. `dvc dag`

In [ ]:
run('dvc', 'dag')

## 10. Metrics

In [ ]:
run('dvc', 'metrics', 'show')

## 11. `dvc push`

Uploads `data/processed/*`, `models/*` from the DVC cache to the remote. On another machine, `dvc pull` downloads them again.

In [ ]:
run('dvc', 'push')

## 12. Commit to Git

Run these in a terminal to save the pipeline state:

```bash
git add dvc.yaml dvc.lock params.yaml .dvc .gitignore metrics plots README.md requirements.txt
git commit -m "Run DVC pipeline"
git push
```